# Couche -> Silver

DuckDB transforme les données brutes de Bronze et écrit les résultats en Parquet compressé dans le bucket `silver`.

Démarrer RustFS et exécuter le [notebook Bronze](01_bronze.ipynb) avant de lancer les cellules Silver dans l'ordre. La section Open Prices récupère aussi les quatre fichiers bruts utilisés pour ses transformations.

### 1 - Préparer DuckDB et RustFS

Installer les dépendances, activer l'accès S3 de DuckDB et créer le bucket Silver si nécessaire.

In [10]:
%pip install -q boto3 duckdb pandas requests

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlsplit

import boto3
import duckdb
import pandas as pd
import requests
from botocore.config import Config

endpoint = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
access_key = os.environ["S3_ACCESS_KEY"]
secret_key = os.environ["S3_SECRET_KEY"]
bucket_name = "silver"

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

if bucket_name not in {bucket["Name"] for bucket in s3.list_buckets()["Buckets"]}:
    s3.create_bucket(Bucket=bucket_name)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

def sql_literal(value):
    return "'" + value.replace("'", "''") + "'"

s3_url = urlsplit(endpoint)
con.execute(f"""
    CREATE OR REPLACE SECRET rustfs_secret (
        TYPE S3,
        KEY_ID {sql_literal(access_key)},
        SECRET {sql_literal(secret_key)},
        ENDPOINT {sql_literal(s3_url.netloc)},
        URL_STYLE 'path',
        USE_SSL {s3_url.scheme == "https"}
    )
""")

exports = {}

def exporter(source, key):
    destination = f"s3://{bucket_name}/{key}"
    (row_count,) = con.execute(f"""
        COPY {source}
        TO {sql_literal(destination)}
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """).fetchone()
    exports[destination] = row_count

### 2 - Instacart

Nettoyer les commandes et les référentiels avant de reconstruire les associations commande-produit.

#### Commandes

Convertir les identifiants et filtrer les commandes dont le numéro, le créneau horaire ou le groupe est invalide.

In [12]:
con.execute("""
    CREATE OR REPLACE TABLE orders_silver AS
    SELECT
        TRY_CAST(order_id AS BIGINT) AS order_id,
        TRY_CAST(user_id AS BIGINT) AS user_id,
        LOWER(TRIM(eval_set)) AS eval_set,
        TRY_CAST(order_number AS INTEGER) AS order_number,
        TRY_CAST(order_dow AS INTEGER) AS order_dow,
        TRY_CAST(order_hour_of_day AS INTEGER) AS order_hour_of_day,
        TRY_CAST(NULLIF(TRIM(days_since_prior_order), '') AS DOUBLE)
            AS days_since_prior_order
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
    WHERE
        TRY_CAST(order_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(user_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(order_number AS INTEGER) > 0
        AND TRY_CAST(order_dow AS INTEGER) BETWEEN 0 AND 6
        AND TRY_CAST(order_hour_of_day AS INTEGER) BETWEEN 0 AND 23
        AND LOWER(TRIM(eval_set)) IN ('prior', 'train', 'test')
""")

exporter("orders_silver", "instacart/orders.parquet")

#### Catalogue, rayons et départements

Nettoyer les libellés et garder une ligne par identifiant valide.

In [13]:
con.execute("""
    CREATE OR REPLACE TABLE products_silver AS

    SELECT
        TRY_CAST(product_id AS BIGINT) AS product_id,
        TRIM(product_name) AS product_name,
        TRY_CAST(aisle_id AS BIGINT) AS aisle_id,
        TRY_CAST(department_id AS BIGINT) AS department_id

    FROM read_parquet('s3://bronze/instacart/raw/products.parquet')

    WHERE
        TRY_CAST(product_id AS BIGINT) IS NOT NULL
        AND NULLIF(TRIM(product_name), '') IS NOT NULL
        AND TRY_CAST(aisle_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(department_id AS BIGINT) IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(product_id AS BIGINT)
        ORDER BY product_name
    ) = 1
""")

exporter("products_silver", "instacart/products.parquet")

con.execute("""
    CREATE OR REPLACE TABLE aisles_silver AS

    SELECT
        TRY_CAST(aisle_id AS INTEGER) AS aisle_id,
        TRIM(aisle) AS aisle

    FROM read_parquet('s3://bronze/instacart/raw/aisles.parquet')

    WHERE
        TRY_CAST(aisle_id AS INTEGER) IS NOT NULL
        AND NULLIF(TRIM(aisle), '') IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(aisle_id AS INTEGER)
        ORDER BY aisle
    ) = 1
""")

exporter("aisles_silver", "instacart/aisles.parquet")

con.execute("""
    CREATE OR REPLACE TABLE departments_silver AS

    SELECT
        TRY_CAST(department_id AS INTEGER) AS department_id,
        TRIM(department) AS department

    FROM read_parquet('s3://bronze/instacart/raw/departments.parquet')

    WHERE
        TRY_CAST(department_id AS INTEGER) IS NOT NULL
        AND NULLIF(TRIM(department), '') IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(department_id AS INTEGER)
        ORDER BY department
    ) = 1
""")

exporter("departments_silver", "instacart/departments.parquet")

#### Produits commandés

Relier chaque ligne à une commande du groupe correspondant et à un produit du catalogue. Garder les positions positives et les indicateurs de réachat valides.

In [14]:
con.execute("""
    CREATE OR REPLACE TABLE order_products_prior_silver AS
    SELECT
        TRY_CAST(op.order_id AS BIGINT) AS order_id,
        TRY_CAST(op.product_id AS BIGINT) AS product_id,
        TRY_CAST(op.add_to_cart_order AS INTEGER) AS add_to_cart_order,
        TRY_CAST(op.reordered AS INTEGER) AS reordered
    FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet') op
    INNER JOIN orders_silver o
        ON TRY_CAST(op.order_id AS BIGINT) = o.order_id
       AND o.eval_set = 'prior'
    INNER JOIN products_silver p
        ON TRY_CAST(op.product_id AS BIGINT) = p.product_id
    WHERE
        TRY_CAST(op.add_to_cart_order AS INTEGER) > 0
        AND TRY_CAST(op.reordered AS INTEGER) IN (0, 1)
""")

exporter("order_products_prior_silver", "instacart/order_products_prior.parquet")

con.execute("""
    CREATE OR REPLACE TABLE order_products_train_silver AS
    SELECT
        TRY_CAST(op.order_id AS BIGINT) AS order_id,
        TRY_CAST(op.product_id AS BIGINT) AS product_id,
        TRY_CAST(op.add_to_cart_order AS INTEGER) AS add_to_cart_order,
        TRY_CAST(op.reordered AS INTEGER) AS reordered
    FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet') op
    INNER JOIN orders_silver o
        ON TRY_CAST(op.order_id AS BIGINT) = o.order_id
       AND o.eval_set = 'train'
    INNER JOIN products_silver p
        ON TRY_CAST(op.product_id AS BIGINT) = p.product_id
    WHERE
        TRY_CAST(op.add_to_cart_order AS INTEGER) > 0
        AND TRY_CAST(op.reordered AS INTEGER) IN (0, 1)
""")

exporter("order_products_train_silver", "instacart/order_products_train.parquet")

### 3 - Open Food Facts

Garder les produits avec un code et un nom, normaliser les champs texte et convertir le groupe NOVA et la quantité par portion.

#### Produits

Exporter les champs descriptifs et nutritionnels sélectionnés.

In [17]:
con.execute("""
    CREATE OR REPLACE TABLE produits_silver AS
    SELECT
        NULLIF(TRIM(code), '') AS code,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(brands), '') AS brands,
        NULLIF(TRIM(categories_tags), '') AS categories_tags,
        NULLIF(TRIM(allergens_tags), '') AS allergens_tags,
        NULLIF(TRIM(ingredients_text), '') AS ingredients_text,

        CASE
            WHEN TRY_CAST(nova_group AS INTEGER) BETWEEN 1 AND 4
            THEN TRY_CAST(nova_group AS INTEGER)
            ELSE NULL
        END AS nova_group,

        LOWER(NULLIF(TRIM(nutriscore_grade), '')) AS nutriscore_grade,

        TRY_CAST(NULLIF(TRIM(serving_quantity), '') AS DOUBLE)
            AS serving_quantity,

        nutriments,
        NULLIF(TRIM(nutriments_estimated), '') AS nutriments_estimated,
        NULLIF(TRIM(nutrition_data), '') AS nutrition_data,
        NULLIF(TRIM(nutrition_data_per), '') AS nutrition_data_per,
        NULLIF(TRIM(nutrition_data_prepared), '') AS nutrition_data_prepared,
        NULLIF(TRIM(nutrition_data_prepared_per), '') AS nutrition_data_prepared_per

    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
    WHERE
        NULLIF(TRIM(code), '') IS NOT NULL
        AND NULLIF(TRIM(product_name), '') IS NOT NULL
""")

exporter("produits_silver", "openfoodfacts/produits.parquet")

### 4 - Open Prices

Importer les jeux de données détaillés dans Bronze, puis standardiser les prix, les produits, les lieux et les preuves.

#### Importer les fichiers bruts

Télécharger les quatre fichiers JSONL compressés et les déposer dans `bronze/openprices/raw/`. Les fichiers temporaires sont supprimés après leur transfert.

In [18]:
with TemporaryDirectory() as directory:
    for name in ("prices", "proofs", "locations", "products"):
        filename = f"{name}.jsonl.gz"
        local_path = Path(directory) / filename
        url = f"https://prices.openfoodfacts.org/data/{filename}"

        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with local_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    file.write(chunk)

        s3.upload_file(str(local_path), "bronze", f"openprices/raw/{filename}")
        local_path.unlink()

#### Prix

Garder les observations identifiées dont le prix est positif et convertir les montants numériques.

In [19]:
con.execute("""
    CREATE OR REPLACE TABLE prices_silver AS
    SELECT
        id,
        product_id,
        location_id,
        proof_id,
        type,
        NULLIF(TRIM(product_code), '') AS product_code,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(category_tag), '') AS category_tag,
        labels_tags,
        origins_tags,
        TRY_CAST(price AS DOUBLE) AS price,
        price_is_discounted,
        TRY_CAST(price_without_discount AS DOUBLE)
            AS price_without_discount,
        NULLIF(TRIM(discount_type), '') AS discount_type,
        NULLIF(TRIM(price_per), '') AS price_per,
        NULLIF(TRIM(currency), '') AS currency,
        location_osm_id,
        location_osm_type,
        date,
        NULLIF(TRIM(receipt_quantity), '') AS receipt_quantity,
        NULLIF(TRIM(owner_comment), '') AS owner_comment,
        NULLIF(TRIM(owner), '') AS owner,
        NULLIF(TRIM(source), '') AS source,
        tags,
        created,
        updated,
        duplicate_of
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    WHERE
        id IS NOT NULL
        AND TRY_CAST(price AS DOUBLE) > 0
""")

exporter("prices_silver", "openprices/prices.parquet")

#### Produits

Garder les produits identifiés et normaliser leurs libellés et leurs scores.

In [20]:
con.execute("""
    CREATE OR REPLACE TABLE openprices_products_silver AS
    SELECT
        id,
        NULLIF(TRIM(code), '') AS code,
        NULLIF(TRIM(source), '') AS source,
        source_last_synced,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(image_url), '') AS image_url,
        product_quantity,
        NULLIF(TRIM(product_quantity_unit), '') AS product_quantity_unit,
        NULLIF(TRIM(quantity), '') AS quantity,
        categories_tags,
        NULLIF(TRIM(brands), '') AS brands,
        brands_tags,
        labels_tags,
        LOWER(NULLIF(TRIM(nutriscore_grade), '')) AS nutriscore_grade,
        LOWER(NULLIF(TRIM(ecoscore_grade), '')) AS ecoscore_grade,
        CASE
            WHEN nova_group BETWEEN 1 AND 4
            THEN nova_group
            ELSE NULL
        END AS nova_group,
        NULLIF(TRIM(creator), '') AS creator,
        unique_scans_n,
        price_count,
        price_currency_count,
        location_count,
        location_type_osm_country_count,
        user_count,
        proof_count,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

exporter("openprices_products_silver", "openprices/products.parquet")

#### Lieux et preuves

Garder les enregistrements identifiés et convertir les coordonnées et les montants des reçus.

In [21]:
con.execute("""
    CREATE OR REPLACE TABLE locations_silver AS
    SELECT
        id,
        NULLIF(TRIM(osm_brand_logo_url), '') AS osm_brand_logo_url,
        NULLIF(TRIM(type), '') AS type,
        osm_id,
        NULLIF(TRIM(osm_type), '') AS osm_type,
        NULLIF(TRIM(osm_name), '') AS osm_name,
        NULLIF(TRIM(osm_display_name), '') AS osm_display_name,
        NULLIF(TRIM(osm_tag_key), '') AS osm_tag_key,
        NULLIF(TRIM(osm_tag_value), '') AS osm_tag_value,
        osm_tags,
        NULLIF(TRIM(osm_brand), '') AS osm_brand,
        NULLIF(TRIM(osm_address_postcode), '') AS osm_address_postcode,
        NULLIF(TRIM(osm_address_city), '') AS osm_address_city,
        NULLIF(TRIM(osm_address_country), '') AS osm_address_country,
        NULLIF(TRIM(osm_address_country_code), '') AS osm_address_country_code,
        TRY_CAST(osm_lat AS DOUBLE) AS osm_lat,
        TRY_CAST(osm_lon AS DOUBLE) AS osm_lon,
        osm_version,
        osm_version_date,
        NULLIF(TRIM(website_url), '') AS website_url,
        tags,
        price_count,
        user_count,
        product_count,
        proof_count,
        NULLIF(TRIM(source), '') AS source,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

exporter("locations_silver", "openprices/locations.parquet")

con.execute("""
    CREATE OR REPLACE TABLE proofs_silver AS
    SELECT
        id,
        location_id,
        NULLIF(TRIM(file_path), '') AS file_path,
        NULLIF(TRIM(mimetype), '') AS mimetype,
        NULLIF(TRIM(type), '') AS type,
        NULLIF(TRIM(image_md5_hash), '') AS image_md5_hash,
        NULLIF(TRIM(image_thumb_path), '') AS image_thumb_path,
        location_osm_id,
        NULLIF(TRIM(location_osm_type), '') AS location_osm_type,
        date,
        NULLIF(TRIM(currency), '') AS currency,
        receipt_price_count,
        TRY_CAST(receipt_price_total AS DOUBLE) AS receipt_price_total,
        TRY_CAST(receipt_online_delivery_costs AS DOUBLE)
            AS receipt_online_delivery_costs,
        ready_for_price_tag_validation,
        owner_consumption,
        NULLIF(TRIM(owner_comment), '') AS owner_comment,
        draft,
        price_count,
        prediction_count,
        NULLIF(TRIM(owner), '') AS owner,
        NULLIF(TRIM(source), '') AS source,
        tags,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

exporter("proofs_silver", "openprices/proofs.parquet")

### 5 - RappelConso

Exporter les rappels en convertissant les dates et les numéros de version. Les valeurs non convertibles deviennent nulles.

In [22]:
exporter("""
    (
        SELECT * REPLACE (
            TRY_CAST(date_publication AS TIMESTAMPTZ) AS date_publication,
            TRY_CAST(date_debut_commercialisation AS TIMESTAMPTZ) AS date_debut_commercialisation,
            TRY_CAST(date_date_fin_commercialisation AS TIMESTAMPTZ) AS date_date_fin_commercialisation,
            TRY_CAST(date_de_fin_de_la_procedure_de_rappel AS TIMESTAMPTZ) AS date_de_fin_de_la_procedure_de_rappel,
            TRY_CAST(numero_version AS INTEGER) AS numero_version
        )
        FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
    )
""", "rappelconso/rappels.parquet")

### 6 - Bilan des exports

Afficher les destinations et le nombre de lignes écrites par DuckDB pour chaque export réussi.

In [23]:
display(pd.DataFrame(exports.items(), columns=["Fichier Silver", "Lignes exportées"]))

,Fichier Silver,Lignes exportées
0,s3://silver/instacart/orders.parquet,3421083
1,s3://silver/instacart/products.parquet,49688
2,s3://silver/instacart/aisles.parquet,134
3,s3://silver/instacart/departments.parquet,21
4,s3://silver/instacart/order_products_prior.par...,32434489
5,s3://silver/instacart/order_products_train.par...,1384617
6,s3://silver/openfoodfacts/produits.parquet,100
7,s3://silver/openprices/prices.parquet,345806
8,s3://silver/openprices/products.parquet,150354
9,s3://silver/openprices/locations.parquet,7947
